# Homework 2: Distributed LLM Training

**Model:** `EleutherAI/pythia-160m`  
**Dataset:** `wikitext-103-v1`  
**Metric:** Validation Perplexity (PPL)  
**Fixed:** `seed=42`, `seq_len=512`, `global_batch_size=131072` tokens/step, 1 epoch


In [ ]:
import json
import csv
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
LOGS = Path('logs')

def load_results(exp_name: str) -> dict:
    """Load results.json for a given experiment."""
    p = LOGS / exp_name / 'results.json'
    if not p.exists():
        return {}
    with open(p) as f:
        return json.load(f)

def load_log(exp_name: str) -> list[dict]:
    """Load training_log.csv for a given experiment."""
    p = LOGS / exp_name / 'training_log.csv'
    if not p.exists():
        return []
    with open(p) as f:
        return list(csv.DictReader(f))

def fmt(r: dict, key: str, decimals: int = 2) -> str:
    v = r.get(key, None)
    if v is None:
        return '—'
    return f'{float(v):.{decimals}f}'

print('Utilities loaded.')

---
## 1. Single-GPU Baseline


In [ ]:
r_fp32  = load_results('p1-fp32')
r_bf16  = load_results('p1-bf16')
r_bf16ac = load_results('p1-bf16-ac')

print('Task 1 — Single-GPU Results')
print(f"{'Config':<35} {'Val PPL':>10} {'Peak Mem (GB)':>15} {'Throughput (tok/s)':>20}")
print('-' * 82)
for label, r in [('fp32', r_fp32), ('bf16', r_bf16), ('bf16 + activation checkpointing', r_bf16ac)]:
    print(f"{label:<35} {fmt(r,'val_ppl'):>10} {fmt(r,'peak_mem_gb'):>15} {fmt(r,'throughput_tps',0):>20}")

In [ ]:
# Plot training loss curves for Task 1
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, exp in [('fp32', 'p1-fp32'), ('bf16', 'p1-bf16'), ('bf16+AC', 'p1-bf16-ac')]:
    rows = load_log(exp)
    if rows:
        steps = [int(r['step']) for r in rows]
        losses = [float(r['train_loss']) for r in rows]
        tps_vals = [float(r['tps']) for r in rows]
        axes[0].plot(steps, losses, label=label)
        axes[1].plot(steps, tps_vals, label=label)

axes[0].set_title('Training Loss (Task 1)')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].set_title('Throughput tok/s (Task 1)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('tok/s')
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/task1_curves.png', bbox_inches='tight')
plt.show()

### 1.A — Память: fp32 vs bf16 (2 балла)

**Теория.** Pythia-160m имеет ~160M параметров:
- **fp32**: 4 байта × 160M = **0.64 GB** только параметры  
- **bf16**: 2 байта × 160M = **0.32 GB** только параметры  
- Теоретическая разница: **0.32 GB**

**Реальная разница** (из таблицы): $X$ GB

**Почему реальная разница $X > 0.32$ GB?**

При обучении в GPU хранятся не только параметры, но и:
1. **Градиенты** — в том же dtype, что и параметры (ещё ×1 от веса).
2. **Optimizer states (Adam)** — в fp32 всегда (momentum + variance = ×2). Для fp32-модели = 2 × 0.64 GB = 1.28 GB. Для bf16-модели — Adam всё равно хранит копию в fp32, итого для оптимизатора в обоих случаях ≈ одинаково!
3. **Активации** — при прямом проходе хранятся тензоры промежуточных вычислений (пропорциональны batch_size × seq_len × hidden_dim × n_layers). В fp32 они в 2× больше, чем в bf16.

Таким образом реальная экономия от bf16 включает:
$$\Delta_{\text{real}} = \underbrace{\Delta_{\text{params}}}_{0.32} + \underbrace{\Delta_{\text{grads}}}_{0.32} + \underbrace{\Delta_{\text{activations}}}_{>0}$$

In [ ]:
# 1.A — Compute and show memory breakdown
n_params = 160e6
fp32_params_gb = 4 * n_params / 1e9
bf16_params_gb = 2 * n_params / 1e9

mem_fp32 = r_fp32.get('peak_mem_gb', None)
mem_bf16 = r_bf16.get('peak_mem_gb', None)

print(f'Theoretical parameter memory fp32: {fp32_params_gb:.3f} GB')
print(f'Theoretical parameter memory bf16: {bf16_params_gb:.3f} GB')
print(f'Theoretical difference:            {fp32_params_gb - bf16_params_gb:.3f} GB')
if mem_fp32 and mem_bf16:
    real_diff = float(mem_fp32) - float(mem_bf16)
    print(f'\nMeasured peak mem fp32:            {float(mem_fp32):.3f} GB')
    print(f'Measured peak mem bf16:            {float(mem_bf16):.3f} GB')
    print(f'Real difference:                   {real_diff:.3f} GB')
    print(f'Overhead beyond params:            {real_diff - (fp32_params_gb - bf16_params_gb):.3f} GB')
    print('  → Explained by: gradients in matching dtype + activations in fp32 taking 2x more memory')

### 1.B — Throughput при activation checkpointing (2 балла)

**Activation checkpointing** (gradient checkpointing) во время forward pass **не сохраняет активации промежуточных слоёв** в памяти GPU. Вместо этого на backward pass они пересчитываются заново.

- **Экономия памяти**: в N раз меньше активаций (где N ≈ число слоёв), что позволяет использовать больший batch size или обучать более крупные модели.
- **Цена**: дополнительный forward pass (частично) → throughput падает примерно на **~20–30%** (ожидаемо).

**Теоретическое обоснование:** для transformer с $L$ слоями без AC хранится $O(L \times B \times S \times H)$ активаций. С AC — $O(\sqrt{L} \times B \times S \times H)$ при оптимальной стратегии сохранения чекпоинтов.


In [ ]:
# 1.B — Throughput comparison
tps_bf16 = float(r_bf16.get('throughput_tps', 0))
tps_bf16ac = float(r_bf16ac.get('throughput_tps', 0))
mem_bf16_v = float(r_bf16.get('peak_mem_gb', 0))
mem_bf16ac_v = float(r_bf16ac.get('peak_mem_gb', 0))

if tps_bf16 > 0 and tps_bf16ac > 0:
    delta_tps = (tps_bf16ac - tps_bf16) / tps_bf16 * 100
    delta_mem = mem_bf16_v - mem_bf16ac_v
    print(f'bf16 throughput:    {tps_bf16:.0f} tok/s')
    print(f'bf16+AC throughput: {tps_bf16ac:.0f} tok/s')
    print(f'Δ throughput:       {delta_tps:+.1f}%  (expected ~-20 to -30%)')
    print(f'\nbf16 peak mem:    {mem_bf16_v:.3f} GB')
    print(f'bf16+AC peak mem: {mem_bf16ac_v:.3f} GB')
    print(f'Memory saved by AC: {delta_mem:.3f} GB')

### 1.C — Val PPL: совпадают ли? (1 балл)

- **fp32 vs bf16**: незначительное расхождение (< 1 единицы) ожидаемо — bf16 имеет меньшую точность представления, что вносит небольшой числовой шум в градиенты, но не меняет сходимость для таких задач.
- **bf16 vs bf16+AC**: должны совпадать **точно**, так как activation checkpointing — это чисто вычислительная оптимизация, математически эквивалентная стандартному обучению. Он не изменяет ни веса, ни градиенты.
- Расхождение > 1 единицы между fp32 и bf16 нежелательно — может указывать на численную нестабильность при малом learning rate или нехватке warmup.


In [ ]:
# 1.C — PPL comparison
for label, r in [('fp32', r_fp32), ('bf16', r_bf16), ('bf16+AC', r_bf16ac)]:
    ppl = r.get('val_ppl', '—')
    print(f'{label:<25}: val_ppl = {ppl}')

ppl_bf16 = float(r_bf16.get('val_ppl', 0))
ppl_bf16ac = float(r_bf16ac.get('val_ppl', 0))
if ppl_bf16 > 0 and ppl_bf16ac > 0:
    diff = abs(ppl_bf16 - ppl_bf16ac)
    print(f'\n|bf16 PPL - bf16+AC PPL| = {diff:.4f}')
    print('Expected: ≈ 0 (AC is mathematically equivalent)')

---
## 2. FSDP: стратегии шардирования


In [ ]:
r2_no  = load_results('p2-no-shard')
r2_sgo = load_results('p2-shard-grad-op')
r2_fs  = load_results('p2-full-shard')

print('Task 2 — FSDP Sharding Strategies (4 GPU, bf16)')
print(f"{'Strategy':<18} {'Val PPL':>10} {'Peak mem/GPU (GB)':>18} {'Throughput (tok/s)':>20}")
print('-' * 70)
for label, r in [('NO_SHARD (DDP)', r2_no), ('SHARD_GRAD_OP', r2_sgo), ('FULL_SHARD', r2_fs)]:
    print(f"{label:<18} {fmt(r,'val_ppl'):>10} {fmt(r,'peak_mem_gpu_gb'):>18} {fmt(r,'throughput_tps',0):>20}")

### 2.A — Теоретический расход памяти (2 балла)

Параметры модели: $P = 160M$. Обучение с bf16, optimizer states в fp32.

**Состояния на шаге обучения:**

| Что | dtype | Размер |
|---|---|---|
| Параметры | bf16 | $2P$ байт |
| Градиенты | bf16/fp32 | $2P$ байт |
| Adam momentum | fp32 | $4P$ байт |
| Adam variance | fp32 | $4P$ байт |
| **Итого model states** | | **$12P$ байт** |

Для $P = 160\text{M}$: $12 \times 160\text{M} = 1.92$ GB (без активаций).

**По стратегиям на 4 GPU:**

- **NO_SHARD (DDP)**: полная копия на каждой GPU → **≈ 1.92 GB** model states/GPU
- **SHARD_GRAD_OP**: шардируются только gradients + optimizer states, параметры реплицированы во время forward → **≈ params(2P) + (grads+optim)/4 = 0.32 + 1.60/4 = 0.32 + 0.40 ≈ 0.72 GB**
- **FULL_SHARD**: всё шардируется → **≈ 12P/4 = 0.48 GB** model states/GPU

Реальные значения выше из-за активаций, буферов аллокатора, CUDA context (≈ 0.3–0.8 GB overhead).


In [ ]:
# 2.A — Theoretical vs measured memory
n_params = 160e6
n_gpus = 4

# bytes per param: 2(param bf16) + 2(grad bf16) + 4(mom fp32) + 4(var fp32) = 12
bytes_per_param_total = 12
total_model_states_gb = bytes_per_param_total * n_params / 1e9

# Theoretical per-GPU
theo = {
    'NO_SHARD':       total_model_states_gb,          # fully replicated
    'SHARD_GRAD_OP':  (2 * n_params / 1e9) + ((2+4+4) * n_params / 1e9) / n_gpus,  # params replicated, rest sharded
    'FULL_SHARD':     total_model_states_gb / n_gpus,  # everything sharded
}

print('Memory analysis (model states only, no activations):')
print(f"{'Strategy':<18} {'Theoretical (GB)':>18} {'Measured (GB)':>15} {'Overhead (GB)':>15}")
print('-' * 70)
for label, (key, r) in [('NO_SHARD', ('NO_SHARD', r2_no)),
                         ('SHARD_GRAD_OP', ('SHARD_GRAD_OP', r2_sgo)),
                         ('FULL_SHARD', ('FULL_SHARD', r2_fs))]:
    t = theo[key]
    m = float(r.get('peak_mem_gpu_gb', 0))
    overhead = m - t if m > 0 else float('nan')
    print(f"{label:<18} {t:>18.3f} {m:>15.3f} {overhead:>15.3f}")

### 2.B — Throughput FULL_SHARD vs NO_SHARD (2 балла)

**Коммуникации в каждой стратегии:**

| Стратегия | Forward | Backward | Optimizer step |
|---|---|---|---|
| NO_SHARD (DDP) | нет | AllReduce(grads) 1× | локально |
| SHARD_GRAD_OP | AllGather(params) 1× | ReduceScatter(grads) | локально |
| FULL_SHARD | AllGather(params) на каждом слое | ReduceScatter(grads) на каждом слое | локально |

FULL_SHARD выполняет **2× AllGather + ReduceScatter на каждый слой** (forward + backward), тогда как DDP — только 1 AllReduce по всем параметрам сразу. Для малых моделей (160M) overhead коммуникаций при FULL_SHARD относительно велик.

**Для модели в 10× больше**: затраты на коммуникации растут линейно с числом параметров, но вычисления растут быстрее (из-за увеличения ширины слоёв), поэтому **relative overhead шардирования уменьшается** — FULL_SHARD становится выгоднее.


In [ ]:
# 2.B — Throughput ratio
tps_no   = float(r2_no.get('throughput_tps', 0))
tps_sgo  = float(r2_sgo.get('throughput_tps', 0))
tps_full = float(r2_fs.get('throughput_tps', 0))

if tps_no > 0 and tps_full > 0:
    ratio = tps_full / tps_no
    print(f'NO_SHARD throughput:   {tps_no:.0f} tok/s')
    print(f'SHARD_GRAD_OP:         {tps_sgo:.0f} tok/s')
    print(f'FULL_SHARD throughput: {tps_full:.0f} tok/s')
    print(f'FULL_SHARD / NO_SHARD: {ratio:.3f}x  ({(ratio-1)*100:+.1f}%)')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

strategies = ['NO_SHARD\n(DDP)', 'SHARD_GRAD_OP', 'FULL_SHARD']
mems   = [float(r.get('peak_mem_gpu_gb', 0)) for r in [r2_no, r2_sgo, r2_fs]]
tps_all = [float(r.get('throughput_tps', 0)) for r in [r2_no, r2_sgo, r2_fs]]
colors = ['#4C72B0', '#DD8452', '#55A868']

axes[0].bar(strategies, mems, color=colors, alpha=0.8, edgecolor='k')
axes[0].set_title('Peak mem/GPU — Task 2 (4 GPU)')
axes[0].set_ylabel('GB')

axes[1].bar(strategies, tps_all, color=colors, alpha=0.8, edgecolor='k')
axes[1].set_title('Throughput — Task 2 (4 GPU)')
axes[1].set_ylabel('tok/s')

plt.tight_layout()
plt.savefig('plots/task2_bars.png', bbox_inches='tight')
plt.show()

### 2.C — Val PPL совпадают? (1 балл)

Теоретически val PPL должны совпадать: все стратегии обеспечивают математически эквивалентное обучение с одинаковым `global_batch_size`, `seed` и `optimizer`. Расхождение может возникнуть из-за:
- Численных различий bf16 при разных порядках операций (маловероятно)
- Разного порядка батчей при разном `world_size` и `DistributedSampler`


In [ ]:
for label, r in [('NO_SHARD', r2_no), ('SHARD_GRAD_OP', r2_sgo), ('FULL_SHARD', r2_fs)]:
    print(f'{label:<20}: val_ppl = {r.get("val_ppl", "—")}')

---
## 3. CPU Offload


In [ ]:
r3_fs    = load_results('p3-full-shard')
r3_co    = load_results('p3-cpu-offload')
r3_coac  = load_results('p3-cpu-offload-ac')

print('Task 3 — CPU Offload (2 GPU, FULL_SHARD, bf16)')
print(f"{'Config':<40} {'Peak GPU mem (GB)':>18} {'Throughput (tok/s)':>20}")
print('-' * 80)
configs = [
    ('FULL_SHARD', r3_fs),
    ('FULL_SHARD + CPUOffload', r3_co),
    ('FULL_SHARD + CPUOffload + AC', r3_coac),
]
for label, r in configs:
    print(f"{label:<40} {fmt(r,'peak_mem_gpu_gb'):>18} {fmt(r,'throughput_tps',0):>20}")

### 3.A — Что делает CPUOffload? (2 балла)

**CPUOffload** перемещает **optimizer states (momentum + variance) и параметры** из GPU RAM в CPU RAM:

- **На CPU постоянно:** optimizer states (fp32 momentum + variance = 8P байт)
- **На GPU только во время forward/backward:** параметры (переносятся на GPU, используются, переносятся обратно)
- **Всегда на GPU:** активации текущего слоя, градиенты

Это даёт экономию ≈ 8P байт = 1.28 GB для 160M модели.

### 3.B — Совместный эффект CPUOffload + AC (2 балла)

Экономия **аддитивна** лишь частично:
- CPUOffload экономит **optimizer states** (8P байт, хранятся на CPU)
- AC экономит **активации** (≈ batch_size × seq_len × hidden × n_layers × 2 байта)

Эти два пула памяти **независимы**, поэтому эффекты **приблизительно аддитивны**. Небольшое отклонение может быть из-за буферов аллокатора и накладных расходов передачи данных CPU↔GPU.

### 3.C — Throughput с CPUOffload (1 балл)

Throughput **падает** (ожидаемо) из-за:
- CPU↔GPU трафика для параметров на каждом forward шаге
- CPU↔GPU трафика для optimizer states на каждом optimizer step
- Пропускная способность PCIe (≈ 16–32 GB/s) значительно ниже NVLink/HBM


In [ ]:
# 3.A/B/C — Memory and throughput analysis
mem_fs   = float(r3_fs.get('peak_mem_gpu_gb', 0))
mem_co   = float(r3_co.get('peak_mem_gpu_gb', 0))
mem_coac = float(r3_coac.get('peak_mem_gpu_gb', 0))

tps_fs   = float(r3_fs.get('throughput_tps', 0))
tps_co   = float(r3_co.get('throughput_tps', 0))
tps_coac = float(r3_coac.get('throughput_tps', 0))

if mem_fs and mem_co:
    print(f'Memory saved by CPUOffload: {mem_fs - mem_co:.3f} GB')
    print(f'  Expected (optimizer states only): {8 * 160e6 / 1e9 / 2:.3f} GB (on 2 GPU)')
if mem_co and mem_coac:
    print(f'Additional memory saved by +AC:    {mem_co - mem_coac:.3f} GB')
if tps_fs and tps_co:
    print(f'\nThroughput change with CPUOffload: {(tps_co/tps_fs - 1)*100:+.1f}%')
if tps_co and tps_coac:
    print(f'Throughput change with +AC:        {(tps_coac/tps_co - 1)*100:+.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = ['FULL_SHARD', 'FULL_SHARD\n+CPUOffload', 'FULL_SHARD\n+CPUOffload+AC']
axes[0].bar(labels, [mem_fs, mem_co, mem_coac], color=['#4C72B0','#DD8452','#55A868'],
            alpha=0.8, edgecolor='k')
axes[0].set_title('Peak GPU Memory — Task 3 (2 GPU)')
axes[0].set_ylabel('GB')

axes[1].bar(labels, [tps_fs, tps_co, tps_coac], color=['#4C72B0','#DD8452','#55A868'],
            alpha=0.8, edgecolor='k')
axes[1].set_title('Throughput — Task 3 (2 GPU)')
axes[1].set_ylabel('tok/s')

plt.tight_layout()
plt.savefig('plots/task3_bars.png', bbox_inches='tight')
plt.show()

---
## 4. Масштабирование по числу GPU


In [ ]:
r4_1 = load_results('p4-1gpu')
r4_2 = load_results('p4-2gpu')
r4_4 = load_results('p4-4gpu')

print('Task 4 — GPU Scaling (FULL_SHARD, bf16)')
print(f"{'N GPU':>6} {'Throughput total (tok/s)':>26} {'Throughput per GPU (tok/s)':>28} {'Peak mem/GPU (GB)':>18}")
print('-' * 82)
for n, r in [(1, r4_1), (2, r4_2), (4, r4_4)]:
    tps_total = float(r.get('throughput_tps', 0))
    tps_per = float(r.get('throughput_per_gpu_tps', tps_total / n if tps_total else 0))
    mem = fmt(r, 'peak_mem_gpu_gb')
    print(f"{n:>6} {tps_total:>26.0f} {tps_per:>28.0f} {mem:>18}")

### 4.A — Scaling efficiency (3 балла)

**Scaling efficiency:**
$$\eta(N) = \frac{\text{throughput\_per\_gpu}(N)}{\text{throughput\_per\_gpu}(1)}$$

Идеальное масштабирование: $\eta(N) = 1.0$. На практике $\eta < 1$ из-за:
- **Коммуникационного overhead** (AllGather + ReduceScatter при FULL_SHARD)
- **Синхронизации** (`dist.barrier()`, NCCL collective overhead)
- **Неравномерности нагрузки** (tail latency)

Для 160M (маленькая модель): коммуникации составляют значительную долю → $\eta$ может быть 0.7–0.9.
Для 7B+: коммуникации накрываются вычислениями → $\eta$ ближе к 1.0.


In [ ]:
# 4.A — Scaling efficiency
tps_1gpu = float(r4_1.get('throughput_per_gpu_tps',
                           r4_1.get('throughput_tps', 0)))
tps_2gpu = float(r4_2.get('throughput_per_gpu_tps',
                           float(r4_2.get('throughput_tps', 0)) / 2))
tps_4gpu = float(r4_4.get('throughput_per_gpu_tps',
                           float(r4_4.get('throughput_tps', 0)) / 4))

n_list = [1, 2, 4]
tps_list = [tps_1gpu, tps_2gpu, tps_4gpu]
eta_list = [t / tps_1gpu if tps_1gpu else 0 for t in tps_list]

for n, tps, eta in zip(n_list, tps_list, eta_list):
    print(f'N={n}: throughput_per_gpu={tps:.0f} tok/s  η={eta:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(n_list, tps_list, 'o-', color='#4C72B0', lw=2, ms=8, label='Measured')
if tps_1gpu:
    axes[0].axhline(tps_1gpu, color='gray', ls='--', label='Ideal (constant)')
axes[0].set_title('Throughput per GPU vs N (Task 4)')
axes[0].set_xlabel('Number of GPUs')
axes[0].set_ylabel('tok/s per GPU')
axes[0].set_xticks(n_list)
axes[0].legend()

axes[1].plot(n_list, eta_list, 's-', color='#DD8452', lw=2, ms=8)
axes[1].axhline(1.0, color='gray', ls='--', label='Perfect scaling η=1')
axes[1].set_title('Scaling Efficiency η(N) (Task 4)')
axes[1].set_xlabel('Number of GPUs')
axes[1].set_ylabel('η = tps_per_gpu(N) / tps_per_gpu(1)')
axes[1].set_xticks(n_list)
axes[1].set_ylim(0, 1.1)
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/task4_scaling.png', bbox_inches='tight')
plt.show()

### 4.B — Memory scaling (2 балла)

При идеальном шардировании `mem(N=1) / mem(N=4)` должно равняться 4. На практике это не так, потому что:

1. **Активации** — не шардируются в FSDP по умолчанию (каждая GPU держит свои активации для своего батча).
2. **CUDA context / буферы NCCL** — фиксированный overhead ~300–500 MB на GPU.
3. **Буферы AllGather** — временные буферы для сборки параметров из шардов.

В итоге: model states масштабируются как 1/N, активации — не масштабируются, overhead — константный.


In [ ]:
# 4.B — Memory scaling analysis
mem_1 = float(r4_1.get('peak_mem_gpu_gb', 0))
mem_2 = float(r4_2.get('peak_mem_gpu_gb', 0))
mem_4 = float(r4_4.get('peak_mem_gpu_gb', 0))

if mem_1 and mem_4:
    ratio_14 = mem_1 / mem_4
    print(f'mem(N=1) = {mem_1:.3f} GB')
    print(f'mem(N=2) = {mem_2:.3f} GB')
    print(f'mem(N=4) = {mem_4:.3f} GB')
    print(f'\nmem(1)/mem(4) = {ratio_14:.2f}x  (ideal: 4x)')
    non_sharded_estimate = mem_1 - (mem_1 - mem_4) * 4 / 3
    print(f'\nEstimated non-sharded memory (activations + CUDA overhead): {non_sharded_estimate:.3f} GB')

fig, ax = plt.subplots(figsize=(7, 4))
ns = [1, 2, 4]
mems = [mem_1, mem_2, mem_4]
ideal = [mem_1 / n for n in ns]

ax.plot(ns, mems, 'o-', color='#4C72B0', lw=2, ms=8, label='Measured')
ax.plot(ns, ideal, 's--', color='gray', lw=1.5, ms=6, label='Ideal (1/N scaling)')
ax.set_title('Peak Memory/GPU vs N GPUs (Task 4)')
ax.set_xlabel('Number of GPUs')
ax.set_ylabel('GB')
ax.set_xticks(ns)
ax.legend()
plt.tight_layout()
plt.savefig('plots/task4_memory.png', bbox_inches='tight')
plt.show()

---
## Bonus A — Влияние размера модели на overhead шардирования


In [ ]:
# Load bonus results: 160m (from task 2) and 410m
b410_no  = load_results('bonus-410m-no-shard')
b410_sgo = load_results('bonus-410m-shard-grad-op')
b410_fs  = load_results('bonus-410m-full-shard')

# Reuse task 2 results for 160m
b160_no  = r2_no
b160_sgo = r2_sgo
b160_fs  = r2_fs

print('Bonus A — 4 GPU comparison: pythia-160m vs pythia-410m')
strategies = ['NO_SHARD', 'SHARD_GRAD_OP', 'FULL_SHARD']
rows_160 = [b160_no, b160_sgo, b160_fs]
rows_410 = [b410_no, b410_sgo, b410_fs]

print(f"\n{'Strategy':<18} {'160m mem':>10} {'410m mem':>10} {'160m tps':>10} {'410m tps':>10}")
print('-' * 62)
for s, r160, r410 in zip(strategies, rows_160, rows_410):
    print(f"{s:<18} {fmt(r160,'peak_mem_gpu_gb'):>10} {fmt(r410,'peak_mem_gpu_gb'):>10} "
          f"{fmt(r160,'throughput_tps',0):>10} {fmt(r410,'throughput_tps',0):>10}")

### A.A — Для какой модели overhead FULL_SHARD больше? (3 балла)

**Гипотеза**: относительный overhead FULL_SHARD больше для **160m**, чем для **410m**.

**Почему**: При FULL_SHARD каждый AllGather пересылает весь шард параметров, а вычисления пропорциональны $O(\text{params}^{2/3})$ (грубо). Для маленькой модели отношение коммуникации к вычислению больше — нельзя перекрыть коммуникации вычислениями. Для 410m каждый слой шире → больше FLOPs → лучше hardware utilization → relative overhead меньше.

### A.B — Экономия памяти NO_SHARD → FULL_SHARD (3 балла)

Теоретически экономия = model_states × (1 - 1/4) = model_states × 0.75.
Для 410m экономия должна быть **пропорционально больше** (в ~2.5× больше параметров).

Однако **соотношение экономии не совпадает точно** с отношением размеров, потому что:
- Активации — зависят от `batch_size × seq_len × hidden_dim`, а не от общего числа параметров
- CUDA/NCCL overhead — константный
- Embeddings (vocab × hidden) — не масштабируются линейно


In [ ]:
# Bonus A — Comparative plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(strategies))
w = 0.35

# Peak mem comparison
mems_160 = [float(r.get('peak_mem_gpu_gb', 0)) for r in rows_160]
mems_410 = [float(r.get('peak_mem_gpu_gb', 0)) for r in rows_410]

axes[0].bar(x - w/2, mems_160, w, label='pythia-160m', color='#4C72B0', alpha=0.8, edgecolor='k')
axes[0].bar(x + w/2, mems_410, w, label='pythia-410m', color='#DD8452', alpha=0.8, edgecolor='k')
axes[0].set_title('Peak mem/GPU — 160m vs 410m (4 GPU)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(strategies)
axes[0].set_ylabel('GB')
axes[0].legend()

# Throughput comparison
tps_160 = [float(r.get('throughput_tps', 0)) for r in rows_160]
tps_410 = [float(r.get('throughput_tps', 0)) for r in rows_410]

axes[1].bar(x - w/2, tps_160, w, label='pythia-160m', color='#4C72B0', alpha=0.8, edgecolor='k')
axes[1].bar(x + w/2, tps_410, w, label='pythia-410m', color='#DD8452', alpha=0.8, edgecolor='k')
axes[1].set_title('Throughput — 160m vs 410m (4 GPU)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(strategies)
axes[1].set_ylabel('tok/s')
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/bonus_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# A.A — Relative overhead of FULL_SHARD vs NO_SHARD for each model
for model_label, r_no, r_fs in [
    ('160m', b160_no, b160_fs),
    ('410m', b410_no, b410_fs)
]:
    tps_no = float(r_no.get('throughput_tps', 0))
    tps_fs = float(r_fs.get('throughput_tps', 0))
    if tps_no > 0 and tps_fs > 0:
        overhead = (1 - tps_fs / tps_no) * 100
        print(f'{model_label}: FULL_SHARD overhead = {overhead:.1f}% vs NO_SHARD')
        print(f'         (NO_SHARD: {tps_no:.0f} tok/s  FULL_SHARD: {tps_fs:.0f} tok/s)')

# A.B — Memory savings NO_SHARD -> FULL_SHARD
print()
for model_label, r_no, r_fs in [
    ('160m', b160_no, b160_fs),
    ('410m', b410_no, b410_fs)
]:
    mem_no = float(r_no.get('peak_mem_gpu_gb', 0))
    mem_fs = float(r_fs.get('peak_mem_gpu_gb', 0))
    if mem_no and mem_fs:
        saved = mem_no - mem_fs
        pct = saved / mem_no * 100
        print(f'{model_label}: memory saved NO_SHARD→FULL_SHARD = {saved:.3f} GB ({pct:.1f}%)')

---
## Итоговые таблицы


In [ ]:
print('=' * 80)
print('TASK 1 — Single-GPU Baseline')
print('=' * 80)
print(f"{'Config':<35} {'Val PPL':>10} {'Peak mem (GB)':>14} {'Throughput (tok/s)':>20}")
print('-' * 81)
for label, r in [('fp32', r_fp32), ('bf16', r_bf16), ('bf16 + activation checkpointing', r_bf16ac)]:
    print(f"{label:<35} {fmt(r,'val_ppl'):>10} {fmt(r,'peak_mem_gb'):>14} {fmt(r,'throughput_tps',0):>20}")

print()
print('=' * 80)
print('TASK 2 — FSDP Strategies (4 GPU, bf16)')
print('=' * 80)
print(f"{'Strategy':<18} {'Val PPL':>10} {'Peak mem/GPU':>14} {'Throughput (tok/s)':>20}")
print('-' * 64)
for label, r in [('NO_SHARD', r2_no), ('SHARD_GRAD_OP', r2_sgo), ('FULL_SHARD', r2_fs)]:
    print(f"{label:<18} {fmt(r,'val_ppl'):>10} {fmt(r,'peak_mem_gpu_gb'):>14} {fmt(r,'throughput_tps',0):>20}")

print()
print('=' * 80)
print('TASK 3 — CPU Offload (2 GPU, FULL_SHARD, bf16)')
print('=' * 80)
print(f"{'Config':<38} {'Peak GPU mem (GB)':>18} {'Throughput (tok/s)':>20}")
print('-' * 78)
for label, r in [('FULL_SHARD', r3_fs), ('FULL_SHARD+CPUOffload', r3_co), ('FULL_SHARD+CPUOffload+AC', r3_coac)]:
    print(f"{label:<38} {fmt(r,'peak_mem_gpu_gb'):>18} {fmt(r,'throughput_tps',0):>20}")

print()
print('=' * 80)
print('TASK 4 — Scaling (FULL_SHARD, bf16)')
print('=' * 80)
print(f"{'N GPU':>6} {'Throughput total':>18} {'Throughput/GPU':>16} {'Peak mem/GPU':>14}")
print('-' * 58)
for n, r in [(1, r4_1), (2, r4_2), (4, r4_4)]:
    tps_total = float(r.get('throughput_tps', 0))
    tps_per = float(r.get('throughput_per_gpu_tps', tps_total / n if tps_total else 0))
    print(f"{n:>6} {tps_total:>18.0f} {tps_per:>16.0f} {fmt(r,'peak_mem_gpu_gb'):>14}")